In [ ]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from datetime import datetime
import copy
import collections
from astropy.cosmology import Planck13  # Planck (2014) Paper XVI, Table 5

import gc
gc.enable()

import sys
sys.path.append('..') 

from utils import Vcom, get_comoving_distance_h, c, z_min
# from data_stat_func import wprp, dPdN
from data_stat_func import weighted_wprp
from data_stat_func import wprp

from halotools.mock_observables import apply_zspace_distortion
from halotools.mock_observables import return_xyz_formatted_array
import halotools.mock_observables as htmo

import Corrfunc
from Corrfunc.mocks.DDrppi_mocks import DDrppi_mocks
from Corrfunc.utils import convert_rp_pi_counts_to_wp
import matplotlib.pyplot as plt
import copy

from Ng8_func import return_box_rsd_pos


In [ ]:
z_max_arr  = np.array((       47650, 31900, 19250, 12500))/ c
M_thr_arr  = np.array((-22.0, -21.0, -20.0, -19.0, -18.0))
C_thr_arr  = np.array([[0.0,  0.82,  0.94,  0.97,   1.2,],
                       [0.0,  0.72,  0.90,  0.94,   1.2,],
                       [0.0,  0.55,  0.85,  0.92,   1.2,],
                       [0.0,  0.50,  0.73,  0.87,   1.2,],])

M_thr_list  = [ 'Mr_22_21', 'Mr_21_20', 'Mr_20_19', 'Mr_19_18', ]
C_thr_list  = np.array([['0_p82'  , 'p82_p94', 'p94_p97', 'p97_1p2',],
                        ['0_p72'  , 'p72_p90', 'p90_p94', 'p94_1p2',],
                        ['0_p55'  , 'p55_p85', 'p85_p92', 'p92_1p2',],
                        ['0_p50'  , 'p50_p73', 'p73_p87', 'p87_1p2',],])

Mr_thr_text = [r'$M_{\rm r}: [-22,-21]$',r'$M_{\rm r}: [-21,-20]$',r'$M_{\rm r}: [-20,-19]$',r'$M_{\rm r}: [-19,-18]$',]
Co_thr_text = np.array([[r'g-r: [0,0.82]', r'g-r: [0.82,0.94]', r'g-r: [0.94,0.97]', r'g-r: [0.97,1.2]',],
                        [r'g-r: [0,0.72]', r'g-r: [0.72,0.90]', r'g-r: [0.90,0.94]', r'g-r: [0.94,1.2]',],
                        [r'g-r: [0,0.55]', r'g-r: [0.55,0.85]', r'g-r: [0.85,0.92]', r'g-r: [0.92,1.2]',],
                        [r'g-r: [0,0.50]', r'g-r: [0.50,0.73]', r'g-r: [0.73,0.87]', r'g-r: [0.87,1.2]',],])

Mr_cross_list = ['2221_2120', '2221_2019', '2221_1918', '2120_2019']

In [ ]:
### Comoving pi_max and wp bins in Mpc/h
rp_bins = np.logspace(-0.8, 1.6, 13)
rp_bins_x = (rp_bins[1:] + rp_bins[:-1]) / 2
# rp_bins_x

from matplotlib import cm
colors4 = [cm.YlGnBu(55)  , cm.YlGnBu(80)  , cm.YlGnBu(120) , cm.YlGnBu(150) , cm.YlGnBu(180)]

In [ ]:
rp_bins

### Cross-correlation

In [ ]:
def DD_rp_pi_to_DD_rp(DD_rp_pi, rp_bins):
    len_rp = len(rp_bins) - 1
    len_pi = len(DD_rp_pi) // len_rp
    DD_rp  = DD_rp_pi.reshape(len_rp, len_pi).sum(axis = 1)
    return DD_rp

In [ ]:
def cross_correlation_wprp(   RA1, DEC1, CZ1, rand_RA1, rand_DEC1, rand_CZ1, 
                              RA2, DEC2, CZ2, rand_RA2, rand_DEC2, rand_CZ2,
                              pi_max, rp_bins, gal_weights = 1.0, random_weights = 1.0, nthread = 55):
      '''
      calculate the cross correlation of D1, D2, R1, R2 using Landy-Szalay estimator
      '''
      #  estimator == "Landy-Szalay":
      ND1 = len(RA1)
      ND2 = len(RA2)
      NR1 = len(rand_RA1)
      NR2 = len(rand_RA2)

      factor1 = ND1 * ND2 / (NR1 * NR2)
      factor2 = ND1 * NR2 / (NR1 * NR2)
      factor3 = ND2 * NR1 / (NR1 * NR2)

      # (D1D2 - D1R2 - D2R1 + R1R2)/R1R2
      D1D2 = DDrppi_mocks(False, 3, nthread, pi_max, rp_bins,
                              copy.deepcopy(RA1), copy.deepcopy(DEC1),
                              get_comoving_distance_h(copy.deepcopy(CZ1)),
                              weights1 = gal_weights,
                              RA2      = copy.deepcopy(RA2),
                              DEC2     = copy.deepcopy(DEC2),
                              CZ2      = get_comoving_distance_h(copy.deepcopy(CZ2)),
                              weights2 = random_weights,
                              is_comoving_dist = True,
                              weight_type = 'pair_product')['npairs']
      
      D1R2 = DDrppi_mocks(False, 3, nthread, pi_max, rp_bins,
                              copy.deepcopy(RA1), copy.deepcopy(DEC1),
                              get_comoving_distance_h(copy.deepcopy(CZ1)),
                              weights1 = gal_weights,
                              RA2      = copy.deepcopy(rand_RA2),
                              DEC2     = copy.deepcopy(rand_DEC2),
                              CZ2      = get_comoving_distance_h(copy.deepcopy(rand_CZ2)),
                              weights2 = random_weights,
                              is_comoving_dist = True,
                              weight_type = 'pair_product')['npairs']
      
      D2R1 = DDrppi_mocks(False, 3, nthread, pi_max, rp_bins,
                              copy.deepcopy(RA2), copy.deepcopy(DEC2),
                              get_comoving_distance_h(copy.deepcopy(CZ2)),
                              weights1 = gal_weights,
                              RA2      = copy.deepcopy(rand_RA1),
                              DEC2     = copy.deepcopy(rand_DEC1),
                              CZ2      = get_comoving_distance_h(copy.deepcopy(rand_CZ1)),
                              weights2 = random_weights,
                              is_comoving_dist = True,
                              weight_type = 'pair_product')['npairs']
      
      R1R2 = DDrppi_mocks(False, 3, nthread, pi_max, rp_bins,
                              copy.deepcopy(rand_RA1), copy.deepcopy(rand_DEC1),
                              get_comoving_distance_h(copy.deepcopy(rand_CZ1)),
                              weights1 = gal_weights,
                              RA2      = copy.deepcopy(rand_RA2),
                              DEC2     = copy.deepcopy(rand_DEC2),
                              CZ2      = get_comoving_distance_h(copy.deepcopy(rand_CZ2)),
                              weights2 = random_weights,
                              is_comoving_dist = True,
                              weight_type = 'pair_product')['npairs']
    
      term1 = (1.0 / factor1) * (D1D2 / R1R2)
      term2 = (1.0 / factor2) * (D1R2 / R1R2)
      term3 = (1.0 / factor3) * (D2R1 / R1R2)
      wp_rp_pi = term1 - term2 - term3 + 1.0
      wp_rp    = DD_rp_pi_to_DD_rp(wp_rp_pi, rp_bins)
    
      return wp_rp

In [ ]:
def cross_correlation_wprp_theory(  X1, Y1, Z1, rand_X1, rand_Y1, rand_Z1, 
                                    X2, Y2, Z2, rand_X2, rand_Y2, rand_Z2,
                                    pi_max, rp_bins, boxsize, gal_weights = 1.0, random_weights = 1.0, nthread = 55):
      '''
      calculate the cross correlation of D1, D2, R1, R2 using Landy-Szalay estimator
      '''
      #  estimator == "Landy-Szalay":
      ND1 = len(X1)
      ND2 = len(X2)
      NR1 = len(rand_X1)
      NR2 = len(rand_X2)

      factor1 = ND1 * ND2 / (NR1 * NR2)
      factor2 = ND1 * NR2 / (NR1 * NR2)
      factor3 = ND2 * NR1 / (NR1 * NR2)

      # (D1D2 - D1R2 - D2R1 + R1R2)/R1R2
      D1D2 = Corrfunc.theory.DDrppi(False, nthread, pi_max, rp_bins,
                                    copy.deepcopy(X1), copy.deepcopy(Y1), copy.deepcopy(Z1),
                                    weights1 = gal_weights,
                                    boxsize  = boxsize,
                                    X2       = copy.deepcopy(X2),
                                    Y2       = copy.deepcopy(Y2),
                                    Z2       = copy.deepcopy(Z2),
                                    weights2 = random_weights,
                                    weight_type = 'pair_product')['npairs']
      
      D1R2 = Corrfunc.theory.DDrppi(False, nthread, pi_max, rp_bins,
                                    copy.deepcopy(X1), copy.deepcopy(Y1), copy.deepcopy(Z1),
                                    weights1 = gal_weights,
                                    boxsize  = boxsize,
                                    X2       = copy.deepcopy(rand_X2),
                                    Y2       = copy.deepcopy(rand_Y2),
                                    Z2       = copy.deepcopy(rand_Z2),
                                    weights2 = random_weights,
                                    weight_type = 'pair_product')['npairs']
      
      D2R1 = Corrfunc.theory.DDrppi(False, nthread, pi_max, rp_bins,
                                    copy.deepcopy(X2), copy.deepcopy(Y2), copy.deepcopy(Z2),
                                    weights1 = gal_weights,
                                    boxsize  = boxsize,
                                    X2       = copy.deepcopy(rand_X1),
                                    Y2       = copy.deepcopy(rand_Y1),
                                    Z2       = copy.deepcopy(rand_Z1),
                                    weights2 = random_weights,
                                    weight_type = 'pair_product')['npairs']
      
      R1R2 = Corrfunc.theory.DDrppi(False, nthread, pi_max, rp_bins,
                                    copy.deepcopy(rand_X1), copy.deepcopy(rand_Y1), copy.deepcopy(rand_Z1),
                                    weights1 = gal_weights,
                                    boxsize  = boxsize,
                                    X2       = copy.deepcopy(rand_X2),
                                    Y2       = copy.deepcopy(rand_Y2),
                                    Z2       = copy.deepcopy(rand_Z2),
                                    weights2 = random_weights,
                                    weight_type = 'pair_product')['npairs']
    
      term1 = (1.0 / factor1) * (D1D2 / R1R2)
      term2 = (1.0 / factor2) * (D1R2 / R1R2)
      term3 = (1.0 / factor3) * (D2R1 / R1R2)
      wp_rp_pi = term1 - term2 - term3 + 1.0
      wp_rp    = DD_rp_pi_to_DD_rp(wp_rp_pi, rp_bins)
    
      return wp_rp

### Mr cross

In [ ]:
### CCMD 32 Quads
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

Mr_cross_list = ['2221_2120', '2221_2019', '2221_1918', '2120_2019']
for var in vars:
    for quad in quads:
        data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data
        output_dict = collections.defaultdict(list)
        for k in range(4):
            stat = []
            if k == 2:
                pi_max = 40
            else:
                pi_max = 60

            if k == 0:
                i = 0 # Mr [-22, -21]
                j = 1 # Mr [-21, -20]
            if k == 1:
                i = 0 # Mr [-22, -21]
                j = 2 # Mr [-20, -19]
            if k == 2:
                i = 0 # Mr [-22, -21]
                j = 3 # Mr [-19, -18]
            if k == 3:
                i = 1 # Mr [-21, -20]
                j = 2 # Mr [-20, -19]
            ##################################################################################################################
            ### Mr mask is related to the z_max 
            mask1       = np.logical_and(   np.logical_and(M_thr_arr[i] < data['Mr'], data['Mr'] < M_thr_arr[i+1]),
                                            np.logical_and(data['z']    >= z_min    , data['z']  < z_max_arr[i]  ),)
            D1_thr      = data[mask1]
            D1_coords   = np.array([np.array(D1_thr['ra']),
                                    np.array(D1_thr['dec']),    
                                    np.array(D1_thr['z'])]).T
            R1_coords   = np.load('./CCMD_32Quads_Mr_bins_Mrp_randoms/CCMD_' 
                                    + var + '_' + quad  + '_' + M_thr_list[i] + '_randoms' + '.npz')['randoms']
            ###################################################################################################################
            mask2       = np.logical_and(   np.logical_and(M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                            np.logical_and(data['z']    >= z_min    , data['z']  < z_max_arr[j]  ),)
            D2_thr      = data[mask2]
            D2_coords   = np.array([np.array(D2_thr['ra']),
                                    np.array(D2_thr['dec']),    
                                    np.array(D2_thr['z'])]).T
            R2_coords   = np.load('./CCMD_32Quads_Mr_bins_Mrp_randoms/CCMD_' 
                                    + var + '_' + quad  + '_' + M_thr_list[j] + '_randoms' + '.npz')['randoms']
            ###################################################################################################################
            wp_rp   = cross_correlation_wprp(   D1_coords[:, 0], D1_coords[:, 1], D1_coords[:, 2], 
                                                R1_coords[:, 0], R1_coords[:, 1], R1_coords[:, 2], 
                                                D2_coords[:, 0], D2_coords[:, 1], D2_coords[:, 2], 
                                                R2_coords[:, 0], R2_coords[:, 1], R2_coords[:, 2],
                                                pi_max = pi_max, rp_bins = rp_bins, 
                                                gal_weights = 1.0, random_weights = 1.0, nthread = 55)
            stat.append(wp_rp)
            
            output_dict['stat_' + Mr_cross_list[k] ] = np.array([item for sublist in stat for item in sublist])

        np.savez('./CCMD_Cross_correlation/Mr_cross/CCMD_' + var + '_' + quad + '_Mr_cross_wprp_pimax60_60_40_60', **output_dict)

In [ ]:
 ### CCMD 32 Quads cov
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

output_dict_stat = collections.defaultdict(list)
output_dict_cov  = collections.defaultdict(list)

for j in range(4): # Mr, z
    data  = []
    for var in vars:
        for quad in quads:
            temp = np.load('./CCMD_Cross_correlation/Mr_cross/CCMD_' + var + '_' + quad + '_Mr_cross_wprp_pimax60_60_40_60.npz')
            data.append(temp['stat_' + Mr_cross_list[j]])

    data  = np.array(data)

    stat = []
    cov  = []
 
    original_statistics = np.mean(data, axis=0)
    cov_matrix          = np.cov(data, rowvar=False)  # rowvar=False, column for variants

    stat.append(original_statistics)
    cov.append(cov_matrix)

    output_dict_stat['stat_' + Mr_cross_list[j] ] = np.array([item for sublist in stat for item in sublist])
    output_dict_cov[ 'cov_'  + Mr_cross_list[j] ] = np.array([item for sublist in cov for item in sublist])

    np.savez('./CCMD_32Quads_Mr_cross_wprp_pimax_60_60_40_60_stat_mean.npz', **output_dict_stat)
    np.savez('./CCMD_32Quads_Mr_cross_wprp_pimax_60_60_40_60_cov.npz', **output_dict_cov)

In [ ]:
wp_rp_CCMD_32Quads     = np.load('./CCMD_32Quads_Mr_cross_wprp_pimax_60_60_40_60_stat_mean.npz')
wp_rp_CCMD_32Quads_cov = np.load('./CCMD_32Quads_Mr_cross_wprp_pimax_60_60_40_60_cov.npz')

wp_rp_bright1_0        = np.load('../CCMD_mag_bin/Cross_correlation/Mr_cross/SDSS_bright1_Mr_2221_2120_cross_wprp_pimax60.npy')
wp_rp_bright1_1        = np.load('../CCMD_mag_bin/Cross_correlation/Mr_cross/SDSS_bright1_Mr_2221_2019_cross_wprp_pimax60.npy')
wp_rp_bright1_2        = np.load('../CCMD_mag_bin/Cross_correlation/Mr_cross/SDSS_bright1_Mr_2221_1918_cross_wprp_pimax40.npy')
wp_rp_bright1_3        = np.load('../CCMD_mag_bin/Cross_correlation/Mr_cross/SDSS_bright1_Mr_2120_2019_cross_wprp_pimax60.npy')
wp_rp_bright1          = [wp_rp_bright1_0     , wp_rp_bright1_1     , wp_rp_bright1_2     , wp_rp_bright1_3     ]
wp_rp_bright1_cov      = np.load('../CCMD_mag_bin/Cross_correlation/Mr_cross/SDSS_bright1_Mr_cross_wprp_pimax60_60_40_60_cov.npz')

Mr_cross_text          = [r'$M_{\rm r} \in [-22, -21] & [-21, -20] $', r'$M_{\rm r} \in [-22, -21] & [-20, -19] $',
                          r'$M_{\rm r} \in [-22, -21] & [-19, -18] $', r'$M_{\rm r} \in [-21, -20] & [-20, -19] $',  ]

In [ ]:
fig, ax = plt.subplots(1, 4, figsize = (20, 4), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

pimax_texts = [r'$\pi_{\rm max} = 60, 60$', r'$\pi_{\rm max} = 60, 60$', r'$\pi_{\rm max} = 40, 40$', r'$\pi_{\rm max} = 60, 60$']

for j in range(4):
    ax.flat[j].plot(rp_bins_x, wp_rp_CCMD_32Quads['stat_' + Mr_cross_list[j] ], lw = 3, ls = '-' , color = 'forestgreen', 
                        label = 'CCMD 32 Quads')
    yerr = np.sqrt(wp_rp_CCMD_32Quads_cov['cov_' + Mr_cross_list[j]].diagonal())
    ax[j].fill_between( rp_bins_x, wp_rp_CCMD_32Quads['stat_' + Mr_cross_list[j] ] - yerr,
                        wp_rp_CCMD_32Quads['stat_' + Mr_cross_list[j] ] + yerr,
                        color = 'forestgreen', alpha = 0.3,)
    yerr = np.sqrt(wp_rp_bright1_cov['cov_' + Mr_cross_list[j]].diagonal())
    ax.flat[j].errorbar(rp_bins_x, wp_rp_bright1[j], yerr = yerr, 
                        mew = 2.5, lw = 2.5, capsize = 0, marker = 'o', ls = 'none', color = 'orange', mfc = 'white', 
                        label = 'SDSS bright1')
#############################################################################################################   
    ax.flat[j].loglog()
    ax.flat[j].tick_params(axis = 'x', direction = 'in', which = u'both')
    ax.flat[j].tick_params(axis = 'y', direction = 'in', which = u'both')
    ax.flat[j].text(0.35, 270, Mr_cross_text[j], fontsize = 15)
    ax.flat[j].text(4.8 , 140, pimax_texts[j]  , fontsize = 15)

#     # ax[i].set_xlim(0.15, 40)
    ax.flat[j].set_ylim(0.5, 500)
    ax.flat[j].set_xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)
ax.flat[0].set_ylabel(r'$w_{\rm p}(r_{\rm p})\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

legend = ax.flat[0].legend(fontsize = 15, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')

# plt.savefig('../../../Figures/fig_data_Mr_C_wp.pdf', bbox_inches='tight')

### Mr thr all & Mr bins bluest (all galaxies in Mr thrs & bluest in Mr bins)
### No thr rands

In [ ]:
# ### CCMD 32 Quads
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

# for var in vars:
#     for quad in quads:
#         data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data
#         output_dict = collections.defaultdict(list)

#         for j in range(4):
#             stat1 = []
#             if j == 3:
#                 pi_max = 40
#             else:
#                 pi_max = 60
#             ##################################################################################################################
#             ### Mr mask is related to the z_max 
#             i = 0 # Mr bin, blue galaxy
#             mask1       = np.logical_and(   np.logical_and( M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
#                                             np.logical_and( np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
#                                                             np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
#             D1_thr      = data[mask1]
#             D1_coords   = np.array([    np.array(D1_thr['ra']),
#                                         np.array(D1_thr['dec']),    
#                                         np.array(D1_thr['z'])]).T
#             # R1_coords   = np.load('./CCMD_32Quads_Mr_bins_Mrp_randoms/CCMD_' 
#                                     # + var + '_' + quad  + '_' + M_thr_list[i] + '_randoms' + '.npz')['randoms']
#             ###################################################################################################################
#             mask2       = np.logical_and(   data['Mr'] < M_thr_arr[j+1],
#                                             np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]           ))
#             D2_thr      = data[mask2]
#             D2_coords   = np.array([    np.array(D2_thr['ra']),
#                                         np.array(D2_thr['dec']),    
#                                         np.array(D2_thr['z'])]).T
#             # R2_coords   = np.load('./CCMD_32Quads_Mr_bins_Mrp_randoms/CCMD_' 
#                                     # + var + '_' + quad  + '_' + M_thr_list[i] + '_randoms' + '.npz')['randoms']
#             ###################################################################################################################
#             if j == 3:
#                 pi_max = 40
#             else:
#                 pi_max = 60
#             wp_rp   = cross_correlation_wprp(   D1_coords[:, 0], D1_coords[:, 1], D1_coords[:, 2], 
#                                                 R1_coords[:, 0], R1_coords[:, 1], R1_coords[:, 2], 
#                                                 D2_coords[:, 0], D2_coords[:, 1], D2_coords[:, 2], 
#                                                 R2_coords[:, 0], R2_coords[:, 1], R2_coords[:, 2],
#                                                 pi_max = pi_max, rp_bins = rp_bins, 
#                                                 gal_weights = 1.0, random_weights = 1.0, nthread = 55)
#             stat1.append(wp_rp)
#             output_dict['wprp_thr_all_bin_bluest' + M_thr_list[j]] = np.array([item for sublist in stat1 for item in sublist])

#         np.savez('./CCMD_Cross_correlation/All_thr_bins_bluest_cross/CCMD_' + var + '_' + quad +  
#                 '_Mr_thr_all_bin_bluest_cross_wprp_pimax60_60_60_40.npz', **output_dict)


In [ ]:
# ### CCMD 32 Quads cov
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

# output_dict_stat = collections.defaultdict(list)
# output_dict_cov  = collections.defaultdict(list)

# for j in range(4): # Mr, z
#     data  = []
#     for var in vars:
#         for quad in quads:
#             temp = np.load('./CCMD_Cross_correlation/All_thr_bins_bluest_cross/CCMD_' + var + '_' + quad +  
#                             '_Mr_thr_all_bin_bluest_cross_wprp_pimax60_60_60_40.npz')
#             data.append(temp['wprp_thr_all_bin_bluest' + M_thr_list[j]])

#     data  = np.array(data)

#     stat = []
#     cov  = []
 
#     original_statistics = np.mean(data, axis=0)
#     cov_matrix          = np.cov(data, rowvar=False)  # rowvar=False, column for variants

#     stat.append(original_statistics)
#     cov.append(cov_matrix)

#     output_dict_stat['stat_' + M_thr_list[j]] = np.array([item for sublist in stat for item in sublist])
#     output_dict_cov[ 'cov_'  + M_thr_list[j]] = np.array([item for sublist in cov for item in sublist])

#     np.savez('./Mr_thr_all_bin_bluest_cross_wprp_pimax60_60_60_40_stat_mean.npz', **output_dict_stat)
#     np.savez('./Mr_thr_all_bin_bluest_cross_wprp_pimax60_60_60_40_cov.npz', **output_dict_cov)

In [ ]:
# wp_rp_bluest_CCMD_32Quads     = np.load('./Mr_thr_all_bin_bluest_cross_wprp_pimax60_60_60_40_stat_mean.npz')
# wp_rp_bluest_CCMD_32Quads_cov = np.load('./Mr_thr_all_bin_bluest_cross_wprp_pimax60_60_60_40_cov.npz')

# wp_rp_red_blue_SDSS_bright1     = np.load('../CCMD_mag_bin/Cross_correlation/SDSS_bright1_Mr_all_blue_cross_wprp_pimax60_60_60_40.npz')

In [ ]:
pimax_thr_text = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
Mr_thr_text2   = ['Mr < -21', 'Mr < -20', 'Mr < -19', 'Mr < -18', ]

fig, ax = plt.subplots(1, 4, figsize = (20, 4), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

for j in range(4):
    # ax[j].plot(rp_bins_x, wp_rp_bluest_CCMD_32Quads['stat_' + M_thr_list[j]], 
            #    color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 32Quads')
    # yerr = np.sqrt(wp_rp_bluest_CCMD_32Quads_cov['cov_'  + M_thr_list[j]].diagonal())
    # ax[j].fill_between(rp_bins_x, wp_rp_bluest_CCMD_32Quads['stat_' + M_thr_list[j]] - yerr, 
                    #    wp_rp_bluest_CCMD_32Quads['stat_' + M_thr_list[j]] + yerr, 
                    #    color = 'forestgreen', alpha = 0.3)
    
    ax[j].errorbar( rp_bins_x, wp_rp_red_blue_SDSS_bright1['wprp_all_blue_' + M_thr_list[j]], 
                    color = 'orangered', alpha = 0.9, mew = 3, marker='o', mfc = 'white',ls='none', label = 'SDSS bright1')
#############################################################################################################   
    ax[j].loglog()
    ax[j].tick_params(axis = 'x', direction = 'in', which = u'both')
    ax[j].tick_params(axis = 'y', direction = 'in', which = u'both')
    ax[j].text(0.4,300, 'bluest:' + Mr_thr_text[j] , fontsize = 14)
    ax[j].text(0.4,150, 'all:' + Mr_thr_text2[j] + ', ' + pimax_thr_text[j], fontsize = 14)

#     # ax[i].set_xlim(0.15, 40)
    ax[j].set_ylim(0.5, 500)
    ax[j].set_xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)
    ax[0].set_ylabel(r'$w_{\rm p}(r_{\rm p})\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

legend = ax[0].legend(fontsize = 12, loc = 'lower left',frameon = False) 
# legend = ax[1].legend(fontsize = 14, loc = 'upper left',frameon = False) 

plt.suptitle('all in Mr_thrs & bluest in Mr_bins', y = 1.0, fontsize = 16)
# plt.savefig('../../../Figures/fig_data_Mr_C_wp.pdf', bbox_inches='tight')

### Mr bins, redest & bluest

In [ ]:
### CCMD 32 Quads
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

for var in vars:
    for quad in quads:
        data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data
        output_dict = collections.defaultdict(list)
        for j in range(4):
            if j == 3:
                pi_max = 40
            else:
                pi_max = 60
                
            stat = []
            ##################################################################################################################
            ### Mr mask is related to the z_max 
            i = 0 # blue galaxy
            mask1       = np.logical_and(   np.logical_and( M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                            np.logical_and( np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
                                                            np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
            D1_thr      = data[mask1]
            D1_coords   = np.array([    np.array(D1_thr['ra']),
                                        np.array(D1_thr['dec']),    
                                        np.array(D1_thr['z'])]).T
            R1_coords = np.load('./CCMD_32Quads_Mr_C_bins_wp_randoms/CCMD_' + var + '_' + quad + '_'
                                        + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '_randoms.npz')['randoms']
            ###################################################################################################################
            i = 3 # red galaxy
            mask2       = np.logical_and(   np.logical_and( M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                            np.logical_and( np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
                                                            np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
            D2_thr      = data[mask2]
            D2_coords   = np.array([    np.array(D2_thr['ra']),
                                        np.array(D2_thr['dec']),    
                                        np.array(D2_thr['z'])]).T
            R2_coords = np.load('./CCMD_32Quads_Mr_C_bins_wp_randoms/CCMD_' + var + '_' + quad + '_'
                                        + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '_randoms.npz')['randoms']
            ###################################################################################################################

            wp_rp   = cross_correlation_wprp(   D1_coords[:, 0], D1_coords[:, 1], D1_coords[:, 2], 
                                                R1_coords[:, 0], R1_coords[:, 1], R1_coords[:, 2], 
                                                D2_coords[:, 0], D2_coords[:, 1], D2_coords[:, 2], 
                                                R2_coords[:, 0], R2_coords[:, 1], R2_coords[:, 2],
                                                pi_max = pi_max, rp_bins = rp_bins, 
                                                gal_weights = 1.0, random_weights = 1.0, nthread = 55)
            stat.append(wp_rp)

            output_dict['wprp_redest_bluest_' + M_thr_list[j]] = np.array([item for sublist in stat for item in sublist])

        np.savez('./CCMD_Cross_correlation/Mr_bins_redest_bluest_cross/CCMD_' + var + '_' + quad  + '_redest_bluest_cross_wprp_pimax60_60_60_40.npz', **output_dict)


In [ ]:
 ### CCMD 32 Quads cov
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

output_dict_stat = collections.defaultdict(list)
output_dict_cov  = collections.defaultdict(list)

for j in range(4): # Mr, z
    data  = []
    for var in vars:
        for quad in quads:
            temp = np.load('./CCMD_Cross_correlation/Mr_bins_redest_bluest_cross/CCMD_' + var + '_' + quad  + '_redest_bluest_cross_wprp_pimax60_60_60_40.npz')
            data.append(temp['wprp_redest_bluest_' + M_thr_list[j]])

    data  = np.array(data)

    stat = []
    cov  = []
 
    original_statistics = np.mean(data, axis=0)
    cov_matrix          = np.cov(data, rowvar=False)  # rowvar=False, column for variants

    stat.append(original_statistics)
    cov.append(cov_matrix)

    output_dict_stat['stat_' + Mr_cross_list[j] ] = np.array([item for sublist in stat for item in sublist])
    output_dict_cov[ 'cov_'  + Mr_cross_list[j] ] = np.array([item for sublist in cov for item in sublist])

    np.savez('./CCMD_32Quads_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_stat_mean.npz', **output_dict_stat)
    np.savez('./CCMD_32Quads_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_cov.npz', **output_dict_cov)

### CCMD 20Vars box, rsd

In [ ]:
### CCMD 20Vars box, rsd
vars = ['Var01', 'Var02', 'Var03', 'Var04', 'Var05', 'Var06', 'Var07', 'Var08', 'Var09', 'Var10', 
        'Var11', 'Var12', 'Var13', 'Var14', 'Var15', 'Var16', 'Var17', 'Var18', 'Var19', 'Var20', ]

for var in vars:
    data = fits.open("/share/hjxu/ccmdmock/Vars/" + var + "/CCMD_Mock_" + var + "_GalHaloInfo.fits")[1].data

    output_dict = collections.defaultdict(list)
    for j in range(4):
        if j == 3:
            pi_max = 40
        else:
            pi_max = 60
            
        stat = []
        ##################################################################################################################
        ### Mr mask is related to the z_max 
        i = 0 # blue galaxy
        mask1       = np.logical_and(   np.logical_and( M_thr_arr[j]   < data['Mr'] , data['Mr']  < M_thr_arr[j+1]),
                                        np.logical_and( C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1]))
        D1_thr      = data[mask1]
        D1_coords = return_box_rsd_pos(D1_thr['x'], D1_thr['y'], D1_thr['z'], D1_thr['vz'],)

        NR = 10*len(D1_thr['x'])
        R1_X = np.random.rand(NR) * 1000
        R1_Y = np.random.rand(NR) * 1000
        R1_Z = np.random.rand(NR) * 1000
        R1_coords = np.array([R1_X, R1_Y, R1_Z]).T
        ###################################################################################################################
        i = 3 # red galaxy
        mask2       = np.logical_and(   np.logical_and( M_thr_arr[j]   < data['Mr'] , data['Mr']  < M_thr_arr[j+1]),
                                        np.logical_and( C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1]))
        D2_thr      = data[mask2]
        D2_coords = return_box_rsd_pos(D2_thr['x'], D2_thr['y'], D2_thr['z'], D2_thr['vz'],)

        NR = 10*len(D2_thr['x'])
        R2_X = np.random.rand(NR) * 1000
        R2_Y = np.random.rand(NR) * 1000
        R2_Z = np.random.rand(NR) * 1000
        R2_coords = np.array([R2_X, R2_Y, R2_Z]).T
        ###################################################################################################################

        wp_rp   = cross_correlation_wprp_theory(    D1_coords[:, 0], D1_coords[:, 1], D1_coords[:, 2], 
                                                    R1_coords[:, 0], R1_coords[:, 1], R1_coords[:, 2], 
                                                    D2_coords[:, 0], D2_coords[:, 1], D2_coords[:, 2], 
                                                    R2_coords[:, 0], R2_coords[:, 1], R2_coords[:, 2],
                                                    pi_max = pi_max, rp_bins = rp_bins, boxsize = 1000, 
                                                    gal_weights = 1.0, random_weights = 1.0, nthread = 55)
        stat.append(wp_rp)

        output_dict['wprp_redest_bluest_' + M_thr_list[j]] = np.array([item for sublist in stat for item in sublist])

    np.savez('./CCMD_20Vars_cross_2pcf/CCMD_' + var + '_box_rsd_redest_bluest_cross_wprp_pimax60_60_60_40.npz', **output_dict)


In [ ]:
### CCMD 20Vars box, rsd, cov
vars = ['Var01', 'Var02', 'Var03', 'Var04', 'Var05', 'Var06', 'Var07', 'Var08', 'Var09', 'Var10', 
        'Var11', 'Var12', 'Var13', 'Var14', 'Var15', 'Var16', 'Var17', 'Var18', 'Var19', 'Var20', ]

output_dict_stat = collections.defaultdict(list)
output_dict_cov  = collections.defaultdict(list)

for j in range(4): # Mr, z
    data  = []
    for var in vars:
        temp = np.load('./CCMD_20Vars_cross_2pcf/CCMD_' + var + '_box_rsd_redest_bluest_cross_wprp_pimax60_60_60_40.npz')
        data.append(temp['wprp_redest_bluest_' + M_thr_list[j]])

    data  = np.array(data)

    output_dict_stat['stat_' + Mr_cross_list[j] ] = np.mean(data, axis=0)
    output_dict_cov[ 'cov_'  + Mr_cross_list[j] ] = np.cov(data, rowvar=False)

    np.savez('./CCMD_20Vars_cross_2pcf/CCMD_20Vars_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_stat_mean.npz', **output_dict_stat)
    np.savez('./CCMD_20Vars_cross_2pcf/CCMD_20Vars_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_cov.npz', **output_dict_cov)

In [ ]:
wp_rp_red_blue_CCMD_32Quads     = np.load('./CCMD_32Quads_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_stat_mean.npz')
wp_rp_red_blue_CCMD_32Quads_cov = np.load('./CCMD_32Quads_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_cov.npz')

wp_rp_red_blue_SDSS_bright1        = np.load('../CCMD_mag_bin/Cross_correlation/SDSS_bright1_Mr_red_blue_cross_wprp_pimax60.npz')
Mr1918_wp_rp_red_blue_SDSS_bright1 = np.load('../CCMD_mag_bin/Cross_correlation/SDSS_bright1_Mr1918_red_blue_cross_wprp_pimax40.npz')
wp_rp_red_blue_SDSS_bright1_cov    = np.load('../CCMD_mag_bin/Cross_correlation/SDSS_bright1_Mr_redest_bluest_cross_wprp_pimax60_60_60_40_cov.npz')

In [ ]:
wp_rp_red_blue_CCMD_20Vars     = np.load('./CCMD_20Vars_cross_2pcf/CCMD_20Vars_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_stat_mean.npz')
wp_rp_red_blue_CCMD_20Vars_cov = np.load('./CCMD_20Vars_cross_2pcf/CCMD_20Vars_Mr_redest_bluest_cross_wprp_pimax_60_60_60_40_cov.npz')

In [ ]:
pimax_thr_text = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
# color_cross_text = [r'$[0, 0.82]\times[0.97, 1.2]$', r'$[0, 0.72]\times[0.94, 1.2]$', r'$[0, 0.55]\times[0.92, 1.2]$', r'$[0, 0.50]\times[0.87, 1.2]$']
color_cross_text = [r'$\times$', r'$\times$', r'$\times$', r'$\times$']
color_blue_text = [r'$[0, 0.82]$', r'$[0, 0.72]$', r'$[0, 0.55]$', r'$[0, 0.50]$']
color_red_text = [r'$[0.97, 1.2]$', r'$[0.94, 1.2]$', r'$[0.92, 1.2]$', r'$[0.87, 1.2]$']

fig, ax = plt.subplots(1, 4, figsize = (20, 4), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

#############################################################################################################   
for j in range(4):
    ### CCMD 32quads ###
    stat = wp_rp_red_blue_CCMD_32Quads['stat_' + Mr_cross_list[j]]
    cov  = wp_rp_red_blue_CCMD_32Quads_cov['cov_'  + Mr_cross_list[j]]
    yerr = np.sqrt( cov.diagonal())
    ax[j].plot( rp_bins_x, stat, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
    ax[j].fill_between( rp_bins_x, stat - yerr, stat + yerr, color = 'forestgreen', alpha = 0.3, lw = 3, ls = '-')
    cov_20Vars = wp_rp_red_blue_CCMD_20Vars_cov['cov_' + Mr_cross_list[j]]
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax[j].fill_between(rp_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err
    ax[j].fill_between(rp_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.3,) # add CCMD 20Vars box err

for j in range(3):
    yerr = np.sqrt(wp_rp_red_blue_SDSS_bright1_cov['cov_' + M_thr_list[j]].diagonal())
    ax[j].errorbar(rp_bins_x, wp_rp_red_blue_SDSS_bright1['wprp_red_blue_' + M_thr_list[j]], yerr = yerr,
                   color = 'orange', alpha = 0.9, mew = 3, marker='o', mfc = 'white',ls='none', label = 'SDSS')
#############################################################################################################   
for j in [3]:
    yerr = np.sqrt(wp_rp_red_blue_SDSS_bright1_cov['cov_' + M_thr_list[j]].diagonal())
    ax[j].errorbar(rp_bins_x, Mr1918_wp_rp_red_blue_SDSS_bright1['wprp_red_blue_' + M_thr_list[j]], yerr = yerr,
                   color = 'orange', alpha = 0.9, mew = 3, marker='o', mfc = 'white',ls='none', label = 'SDSS')
#############################################################################################################
# for j in range(4):
#     ### CCMD 20Vars ###
#     stat = wp_rp_red_blue_CCMD_20Vars['stat_' + Mr_cross_list[j]]     
#     yerr = np.sqrt(wp_rp_red_blue_CCMD_20Vars_cov['cov_' + Mr_cross_list[j]].diagonal())
#     ax[j].plot( rp_bins_x, stat, color = 'royalblue', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 20Vars box w/ rsd')
#     ax[j].fill_between( rp_bins_x, stat - yerr, stat + yerr, color = 'royalblue', alpha = 0.3, lw = 3, ls = '-')
########################################################################################   
for j in range(4): 
    ax[j].loglog()
    ax[j].tick_params(axis = 'x', direction = 'in', which = u'both')
    ax[j].tick_params(axis = 'y', direction = 'in', which = u'both')
    ax[j].text(2.6, 110, Mr_thr_text[j]   , fontsize = 16)
    # ax[j].text(7.5, 120, pimax_thr_text[j], fontsize = 16)
    # ax[j].text(1.2, 120, color_cross_text[j], fontsize = 16)

    add = [3.5,3.5,3.5,3.5]
    ax[j].text(0.65, 220, r'g-r:', fontsize = 16)
    ax[j].text(4.7, 220, color_cross_text[j], fontsize = 16)
    ax[j].text(1.2, 220, color_blue_text[j], fontsize = 16, color = 'royalblue')
    ax[j].text(6.6, 220, color_red_text[j], fontsize = 16, color = 'red')

#     # ax[i].set_xlim(0.15, 40)
    ax[j].set_ylim(0.5, 500)
    ax[j].set_xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)
    ax[0].set_ylabel(r'$w_{\rm p}(r_{\rm p})\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

legend = ax[0].legend(fontsize = 16, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
# legend.get_texts()[1].set_color('royalblue')
legend.get_texts()[1].set_color('orange')
# legend = ax[1].legend(fontsize = 14, loc = 'upper left',frameon = False) 

# plt.suptitle('redest & bluest', y = 1.00, fontsize = 16)
# plt.savefig('../../../Figures/fig_data_Mr_C_wp.pdf', bbox_inches='tight')

### faintest red & brightest blue

In [ ]:
### CCMD Box
main_mock01_path = "/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits"
boxsize = 1000
data    = fits.open(main_mock01_path)[1].data
# hdul[1].header
output_dict1 = collections.defaultdict(list)

### brightest bluest mask 
j = 0
mask1  =  np.logical_and(   np.logical_and(M_thr_arr[j]   < data['Mr'] , data['Mr']  < M_thr_arr[j+1]),
                            np.logical_and(C_thr_arr[j,0] < data['gmr'], data['gmr'] < C_thr_arr[j,1]) )
### faintest redest mask
j = 3
mask2  =  np.logical_and(   np.logical_and(M_thr_arr[j]   < data['Mr'] , data['Mr']  < M_thr_arr[j+1]),
                            np.logical_and(C_thr_arr[j,3] < data['gmr'], data['gmr'] < C_thr_arr[j,4]) )
#########################################################################################################
Data1   = data[mask1]
Data_x1 = Data1['x']% 1000
Data_y1 = Data1['y']% 1000
Data_z1 = apply_zspace_distortion(  true_pos  = Data1['z'], peculiar_velocity = Data1['vz'], redshift = 0, 
                                    cosmology = Planck13  , Lbox = 1000) % 1000 # box size = 1000 Mpc/h
### Randoms
Rand_N1 = 3*len(Data_x1)
seed    = 42
np.random.seed(seed)
Rand_x1 = np.random.uniform(0, boxsize, Rand_N1)
Rand_y1 = np.random.uniform(0, boxsize, Rand_N1)
Rand_z1 = np.random.uniform(0, boxsize, Rand_N1)
#########################################################################################################
Data2   = data[mask2]
Data_x2 = Data2['x']% 1000
Data_y2 = Data2['y']% 1000
Data_z2 = apply_zspace_distortion(  true_pos  = Data2['z'], peculiar_velocity = Data2['vz'], redshift = 0, 
                                    cosmology = Planck13  , Lbox = 1000) % 1000 # box size = 1000 Mpc/h
### Randoms
Rand_N2 = 3*len(Data_x2)
seed    = 42
np.random.seed(seed)
Rand_x2 = np.random.uniform(0, boxsize, Rand_N2)
Rand_y2 = np.random.uniform(0, boxsize, Rand_N2)
Rand_z2 = np.random.uniform(0, boxsize, Rand_N2)
#########################################################################################################

wp_rp   = cross_correlation_wprp_theory(Data_x1, Data_y1, Data_z1, Rand_x1, Rand_y1, Rand_z1, 
                                        Data_x2, Data_y2, Data_z2, Rand_x2, Rand_y2, Rand_z2,
                                        pi_max = pi_max, rp_bins = rp_bins, boxsize = 1000, 
                                        gal_weights = 1.0, random_weights = 1.0, nthread = 55)

# np.save('./Cross_correlation/CCMD_Box_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax60.npy', wp_rp)
np.save('./Cross_correlation/CCMD_Box_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax40.npy', wp_rp)

In [ ]:
#################### 
### CCMD Mock01
paths = {'Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',}
### SDSS bright1
# paths = {'SDSS_bright1'}

for path in paths:
    data = fits.open('../CCMD_main_processed/CCMD_' + path + '_processed_catalog.fits')[1].data # CCMD Mock01
    # data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1
    ##################################################################################################################
    ### Mr mask is related to the z_max 
    j = 0 # brightest bluest galaxy
    i = 0 # brightest bluest galaxy
    mask1       = np.logical_and(   np.logical_and( M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                    np.logical_and( np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
                                                    np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
    D1_thr      = data[mask1]
    D1_coords   = np.array([    np.array(D1_thr['ra']),
                                np.array(D1_thr['dec']),    
                                np.array(D1_thr['z'])]).T
    R1_coords   = np.load('../CCMD_color_mag_bin/wp_rand_shuffle_z_CCMD/' + path + '_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '.npz')['randoms']
    # R1_coords   = np.load('../CCMD_color_mag_bin/wp_rand_shuffle_z_SDSS_bright1/SDSS_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '.npz')['randoms']
    ###################################################################################################################
    j = 3 # faintest redest galaxy
    i = 3 # faintest redest galaxy
    mask2       = np.logical_and(   np.logical_and( M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                    np.logical_and( np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
                                                    np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
    D2_thr      = data[mask2]
    D2_coords   = np.array([    np.array(D2_thr['ra']),
                                np.array(D2_thr['dec']),    
                                np.array(D2_thr['z'])]).T
    R2_coords   = np.load('../CCMD_color_mag_bin/wp_rand_shuffle_z_CCMD/' + path + '_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '.npz')['randoms']
    # R2_coords   = np.load('../CCMD_color_mag_bin/wp_rand_shuffle_z_SDSS_bright1/SDSS_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '.npz')['randoms']
    ###################################################################################################################

    wp_rp   = cross_correlation_wprp(   D1_coords[:, 0], D1_coords[:, 1], D1_coords[:, 2], R1_coords[:, 0], R1_coords[:, 1], R1_coords[:, 2], 
                                        D2_coords[:, 0], D2_coords[:, 1], D2_coords[:, 2], R2_coords[:, 0], R2_coords[:, 1], R2_coords[:, 2],
                                        pi_max = pi_max, rp_bins = rp_bins, 
                                        gal_weights = 1.0, random_weights = 1.0, nthread = 55)

    # # np.save('./Cross_correlation/CCMD_' + path + '_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax60.npy', wp_rp)
    # np.save('./Cross_correlation/SDSS_bright1_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax60.npy', wp_rp)

    
    np.save('./Cross_correlation/CCMD_' + path + '_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax40.npy', wp_rp)
    # np.save('./Cross_correlation/SDSS_bright1_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax40.npy', wp_rp)


In [ ]:
# wp_rp_CCMD_Box   = np.load('./Cross_correlation/CCMD_Box_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax60.npy' )
# wp_rp_CCMD_Quad0 = np.load('./Cross_correlation/CCMD_Main_Quad0_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax60.npy' )
# wp_rp_CCMD_Quad1 = np.load('./Cross_correlation/CCMD_Main_Quad1_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax60.npy' )
# wp_rp_CCMD_Quad2 = np.load('./Cross_correlation/CCMD_Main_Quad2_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax60.npy' )
# wp_rp_CCMD_Quad3 = np.load('./Cross_correlation/CCMD_Main_Quad3_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax60.npy' )
# wp_rp_bright1    = np.load('./Cross_correlation/SDSS_bright1_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax60.npy')

wp_rp_CCMD_Box   = np.load('./Cross_correlation/CCMD_Box_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax40.npy' )
wp_rp_CCMD_Quad0 = np.load('./Cross_correlation/CCMD_Main_Quad0_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax40.npy' )
wp_rp_CCMD_Quad1 = np.load('./Cross_correlation/CCMD_Main_Quad1_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax40.npy' )
wp_rp_CCMD_Quad2 = np.load('./Cross_correlation/CCMD_Main_Quad2_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax40.npy' )
wp_rp_CCMD_Quad3 = np.load('./Cross_correlation/CCMD_Main_Quad3_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax40.npy' )
wp_rp_bright1    = np.load('./Cross_correlation/SDSS_bright1_Mr_faintest_redest_brightest_bluest_cross_wprp_pimax40.npy')

In [ ]:
plt.plot(rp_bins_x, wp_rp_CCMD_Box  , lw = 3, ls = '-' , color = 'purple'  , label = 'CCMD Box')
plt.plot(rp_bins_x, wp_rp_CCMD_Quad0, lw = 3, ls = '--', color = colors4[0], label = 'CCMD Quad0')
plt.plot(rp_bins_x, wp_rp_CCMD_Quad1, lw = 3, ls = '--', color = colors4[1], label = 'CCMD Quad1')
plt.plot(rp_bins_x, wp_rp_CCMD_Quad2, lw = 3, ls = '--', color = colors4[2], label = 'CCMD Quad2')
plt.plot(rp_bins_x, wp_rp_CCMD_Quad3, lw = 3, ls = '--', color = colors4[3], label = 'CCMD Quad3')
# plt.plot(rp_bins_x, wp_rp_bright1   , lw = 3, ls = '--', color = 'orange'  , label = 'SDSS bright1')
plt.errorbar(rp_bins_x, wp_rp_bright1   , mew = 3, marker = 'o',ls = 'none', color = 'orangered'  ,mfc='white' ,label = 'SDSS bright1')

plt.xscale('log')
plt.yscale('log')
plt.tick_params(axis = 'x', direction = 'in', which = u'both')
plt.tick_params(axis = 'y', direction = 'in', which = u'both')
plt.text(2.2,300, r'$Mr \in [-22, -21] & [-19, -18] $', fontsize = 14)
plt.text(10 ,150, r'$\pi_{\rm max} = 40, 40$', fontsize = 14)
 
# plt.xlim(0.15, 40)
plt.ylim(0.5, 500)
plt.xlabel(r'$r_{\rm p}\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)
plt.ylabel(r'$w_{\rm p}(r_{\rm p})\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

plt.title('faintest redest & brightest bluest', y = 1.00, fontsize = 16)

plt.legend(fontsize = 14, loc = 'lower left',frameon = False)